[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [sqlite3, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlite3-deep-dive.html)

# Everyday Requests &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

The first cell builds `scratch/school.db` and `scratch/ledger.db` as the notebook's Setup did, with
`connect`, `transaction`, `show`, `dollars` and `bar`, and creates the `invoice_balances` view the
notebook's worked examples create. Run it first. The tasks do not depend on one another, and the last
cell closes both connections and removes the scratch folder.


In [1]:
import csv
import shutil
import sqlite3
from contextlib import contextmanager
from datetime import datetime, timedelta
from pathlib import Path

SCRATCH = Path("scratch")
shutil.rmtree(SCRATCH, ignore_errors=True)
SCRATCH.mkdir()
SCHOOL = SCRATCH / "school.db"
LEDGER = SCRATCH / "ledger.db"
TODAY = "2026-04-15"          # every report runs as of a date, so that it gives the same answer tomorrow


def connect(path):
    """A connection that enforces foreign keys, returns rows by column name, and writes its own transactions."""
    conn = sqlite3.connect(path, autocommit=True)
    conn.row_factory = sqlite3.Row
    conn.execute("PRAGMA foreign_keys = ON")
    return conn


@contextmanager
def transaction(conn):
    """Run a with block inside BEGIN IMMEDIATE and COMMIT, or ROLLBACK if anything in it raises."""
    conn.execute("BEGIN IMMEDIATE")
    try:
        yield
        conn.execute("COMMIT")
    except BaseException:
        conn.execute("ROLLBACK")
        raise


def show(cursor):
    """Print a query's rows as a table, with the column names cursor.description gives, and say how many."""
    rows = [["" if value is None else f"{value}" for value in row] for row in cursor.fetchall()]
    names = [column[0] for column in cursor.description]
    widths = [max([len(name)] + [len(row[i]) for row in rows]) for i, name in enumerate(names)]
    print("  ".join(name.ljust(width) for name, width in zip(names, widths)))
    print("  ".join("-" * width for width in widths))
    for row in rows:
        print("  ".join(value.ljust(width) for value, width in zip(row, widths)))
    print(f"({len(rows)} rows)")


def dollars(cents):
    """Cents, as the dollars people write in an email."""
    return f"${cents / 100:,.2f}"


def bar(cents, largest, width=32):
    """A bar for a chart, as long as cents is against the largest value in the report."""
    return "#" * round(width * cents / largest) if largest else ""

STUDENT_NAMES = [
    "Ana Reyes", "Ben Okafor", "Chloe Martin", "Daniel Kim", "Elena Petrova", "Felix Wagner",
    "Grace Lin", "Hassan Ali", "Isabel Costa", "Jonas Berg", "Keiko Tanaka", "Liam Murphy",
    "Maya Patel", "Noah Andersen", "Olivia Brandt", "Pavel Novak", "Quinn Harper", "Rosa Delgado",
    "Sam Ito", "Tara Nilsen", "Umar Farouk", "Vera Kowalski", "Wes Carter", "Yara Haddad",
]
PROGRAMS = ["Biology", "Computer Science", "Mathematics", "Psychology", "History"]
COURSES = [
    ("BIO-101", "Introduction to Biology", "Biology", 4),
    ("CHE-110", "General Chemistry", "Chemistry", 4),
    ("MAT-120", "Calculus I", "Mathematics", 4),
    ("MAT-121", "Calculus II", "Mathematics", 4),
    ("CSC-101", "Programming I", "Computer Science", 3),
    ("CSC-201", "Data Structures", "Computer Science", 3),
    ("ENG-105", "Composition", "English", 3),
    ("HIS-110", "World History", "History", 3),
    ("PSY-101", "Introduction to Psychology", "Psychology", 3),
    ("STA-200", "Statistics", "Mathematics", 3),
]
TERMS = [("2025 Spring", "2025-01-13", "2025-05-02"), ("2025 Fall", "2025-08-25", "2025-12-12"),
         ("2026 Spring", "2026-01-12", "2026-05-01")]
GRADES = [("A", 4.0, 1), ("A-", 3.7, 1), ("B+", 3.3, 1), ("B", 3.0, 1), ("B-", 2.7, 1), ("C+", 2.3, 1),
          ("C", 2.0, 1), ("C-", 1.7, 1), ("D", 1.0, 1), ("F", 0.0, 0), ("W", None, 0)]
MEETINGS = [("MWF", "08:00", "08:50"), ("MWF", "09:00", "09:50"), ("TTh", "09:30", "10:45"),
            ("MWF", "11:00", "11:50"), ("TTh", "13:00", "14:15"), ("MWF", "14:00", "14:50")]
INSTRUCTORS = ["Dr. Hale", "Dr. Osei", "Dr. Ibarra", "Dr. Novak", "Dr. Mensah", "Dr. Lindgren"]
ROOMS = ["SCI 210", "SCI 118", "MAT 004", "HUM 302", "LAB 101", "HUM 210"]

CUSTOMERS = [("Northwind Foods", 30), ("Blue Harbor Design", 30), ("Granite Systems", 45),
             ("Pine Ridge Clinic", 30), ("Aster Logistics", 60), ("Kestrel Media", 45),
             ("Lantern Books", 30), ("Vela Robotics", 60)]
CATALOG = [("Consulting, senior hour", 18500), ("Consulting, standard hour", 12500),
           ("Implementation, fixed fee", 145000), ("Support, monthly", 45000),
           ("Training, per seat", 35000), ("Hosting, monthly", 25000)]
METHODS = ["ACH", "check", "card"]

SCHOOL_SCHEMA = """
    CREATE TABLE students (id INTEGER PRIMARY KEY, name TEXT NOT NULL, email TEXT NOT NULL UNIQUE,
                           program TEXT NOT NULL, started_on TEXT NOT NULL) STRICT;
    CREATE TABLE courses (id INTEGER PRIMARY KEY, code TEXT NOT NULL UNIQUE, title TEXT NOT NULL,
                          department TEXT NOT NULL, credits INTEGER NOT NULL) STRICT;
    CREATE TABLE terms (id INTEGER PRIMARY KEY, name TEXT NOT NULL UNIQUE, starts_on TEXT NOT NULL,
                        ends_on TEXT NOT NULL) STRICT;
    CREATE TABLE grades (grade TEXT PRIMARY KEY, points REAL, earns_credit INTEGER NOT NULL) STRICT;
    CREATE TABLE sections (id INTEGER PRIMARY KEY,
                           course_id INTEGER NOT NULL REFERENCES courses (id),
                           term_id INTEGER NOT NULL REFERENCES terms (id),
                           letter TEXT NOT NULL, instructor TEXT NOT NULL, room TEXT NOT NULL,
                           meets TEXT NOT NULL, starts_at TEXT NOT NULL, ends_at TEXT NOT NULL,
                           capacity INTEGER NOT NULL, UNIQUE (course_id, term_id, letter)) STRICT;
    CREATE TABLE enrollments (id INTEGER PRIMARY KEY,
                              student_id INTEGER NOT NULL REFERENCES students (id),
                              section_id INTEGER NOT NULL REFERENCES sections (id),
                              status TEXT NOT NULL CHECK (status IN ('enrolled', 'completed', 'withdrawn')),
                              grade TEXT REFERENCES grades (grade),
                              UNIQUE (student_id, section_id)) STRICT;
    CREATE INDEX enrollments_by_section ON enrollments (section_id);
"""

LEDGER_SCHEMA = """
    CREATE TABLE customers (id INTEGER PRIMARY KEY, name TEXT NOT NULL UNIQUE,
                            terms_days INTEGER NOT NULL, since TEXT NOT NULL) STRICT;
    CREATE TABLE invoices (id INTEGER PRIMARY KEY, number TEXT NOT NULL UNIQUE,
                           customer_id INTEGER NOT NULL REFERENCES customers (id),
                           issued_on TEXT NOT NULL, due_on TEXT NOT NULL,
                           status TEXT NOT NULL CHECK (status IN ('open', 'void'))) STRICT;
    CREATE TABLE invoice_lines (id INTEGER PRIMARY KEY,
                                invoice_id INTEGER NOT NULL REFERENCES invoices (id),
                                description TEXT NOT NULL, quantity INTEGER NOT NULL,
                                unit_cents INTEGER NOT NULL) STRICT;
    CREATE TABLE payments (id INTEGER PRIMARY KEY,
                           customer_id INTEGER NOT NULL REFERENCES customers (id),
                           received_at TEXT NOT NULL, amount_cents INTEGER NOT NULL,
                           method TEXT NOT NULL CHECK (method IN ('ACH', 'check', 'card'))) STRICT;
    CREATE TABLE payment_applications (payment_id INTEGER NOT NULL REFERENCES payments (id),
                                       invoice_id INTEGER NOT NULL REFERENCES invoices (id),
                                       amount_cents INTEGER NOT NULL,
                                       PRIMARY KEY (payment_id, invoice_id)) STRICT;
    CREATE INDEX invoices_by_customer ON invoices (customer_id, issued_on);
"""


def build_school(path):
    """A small college: its students, courses, terms, sections and enrollments, every value from a formula."""
    conn = connect(path)
    conn.executescript(SCHOOL_SCHEMA)
    with transaction(conn):
        conn.executemany("INSERT INTO grades (grade, points, earns_credit) VALUES (?, ?, ?)", GRADES)
        conn.executemany("INSERT INTO terms (name, starts_on, ends_on) VALUES (?, ?, ?)", TERMS)
        conn.executemany("INSERT INTO courses (code, title, department, credits) VALUES (?, ?, ?, ?)", COURSES)
        conn.executemany("INSERT INTO students (name, email, program, started_on) VALUES (?, ?, ?, ?)",
                         [(name, f"{name[0]}{name.split()[1]}@college.edu".lower(),
                           PROGRAMS[i % len(PROGRAMS)], TERMS[i % 2][1]) for i, name in enumerate(STUDENT_NAMES)])
        sections = []
        for t in range(len(TERMS)):
            for c in range(len(COURSES)):
                for letter in (["A", "B"] if (c + t) % 3 == 0 else ["A"]):
                    n = c * 2 + t + (letter == "B")
                    meets, starts_at, ends_at = MEETINGS[n % len(MEETINGS)]
                    sections.append((c + 1, t + 1, letter, INSTRUCTORS[n % len(INSTRUCTORS)],
                                     ROOMS[(c + 2 * t) % len(ROOMS)], meets, starts_at, ends_at,
                                     8 + (c % 3) * 2))
        conn.executemany("""INSERT INTO sections (course_id, term_id, letter, instructor, room, meets,
                            starts_at, ends_at, capacity) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)""", sections)
        by_term = {}
        for row in conn.execute("SELECT id, term_id, course_id FROM sections ORDER BY id"):
            by_term.setdefault(row["term_id"], []).append((row["id"], row["course_id"]))
        enrollments = []
        for t in range(1, len(TERMS) + 1):
            for s in range(1, len(STUDENT_NAMES) + 1):
                taken = {}
                for k in range(3 + (s + t) % 2):
                    section_id, course_id = by_term[t][(s * 5 + t * 3 + k * 7) % len(by_term[t])]
                    taken[course_id] = section_id
                for course_id, section_id in taken.items():
                    if t == len(TERMS):
                        enrollments.append((s, section_id, "enrolled", None))
                    elif (s * 3 + section_id) % 23 == 0:
                        enrollments.append((s, section_id, "withdrawn", "W"))
                    else:
                        enrollments.append((s, section_id, "completed",
                                            GRADES[(s * 7 + course_id * 5 + t * 3) % 10][0]))
        conn.executemany("INSERT INTO enrollments (student_id, section_id, status, grade) VALUES (?, ?, ?, ?)",
                         enrollments)
    with transaction(conn):
        conn.execute("""
            INSERT INTO sections (course_id, term_id, letter, instructor, room, meets, starts_at, ends_at, capacity)
            SELECT courses.id, terms.id, 'B', 'Dr. Lindgren', 'HUM 302', 'MWF', '11:00', '11:50', 8
            FROM courses, terms WHERE courses.code = 'ENG-105' AND terms.name = '2026 Spring'
        """)                                              # opened this week, and nobody has taken it yet
        conn.executemany("INSERT INTO students (name, email, program, started_on) VALUES (?, ?, ?, ?)",
                         [("Mateo Silva", "msilva@college.edu", "History", "2026-01-12"),
                          ("Zoe Iversen", "ziversen@college.edu", "Biology", "2026-01-12")])
        conn.executemany("""
            INSERT INTO enrollments (student_id, section_id, status)
            SELECT (SELECT id FROM students WHERE email = 'ziversen@college.edu'), sections.id, 'enrolled'
            FROM sections
            JOIN courses ON courses.id = sections.course_id
            JOIN terms ON terms.id = sections.term_id
            WHERE courses.code = ? AND sections.letter = ? AND terms.name = '2026 Spring'
        """, [("CSC-101", "A"), ("MAT-120", "A"), ("CHE-110", "B")])   # registered, nothing completed yet
    return conn


def build_ledger(path):
    """Half a year of invoices and payments for eight customers, every value from a formula."""
    conn = connect(path)
    conn.executescript(LEDGER_SCHEMA)
    with transaction(conn):
        conn.executemany("INSERT INTO customers (name, terms_days, since) VALUES (?, ?, ?)",
                         [(name, days, f"202{3 + i % 3}-0{1 + i % 9}-05")
                          for i, (name, days) in enumerate(CUSTOMERS)])
        conn.execute("INSERT INTO customers (name, terms_days, since) VALUES (?, ?, ?)",
                     ("Harbor Lights Studio", 30, "2026-04-06"))     # signed this month, nothing invoiced yet
        for n in range(48):
            day = datetime(2025, 10, 1) + timedelta(days=n * 4)
            customer = n % len(CUSTOMERS) + 1
            due = day + timedelta(days=CUSTOMERS[customer - 1][1])
            invoice_id = conn.execute("""INSERT INTO invoices (number, customer_id, issued_on, due_on, status)
                                         VALUES (?, ?, ?, ?, ?) RETURNING id""",
                                      (f"INV-{1000 + n}", customer, day.strftime("%Y-%m-%d"),
                                       due.strftime("%Y-%m-%d"), "void" if n in (11, 34) else "open")).fetchone()["id"]
            for line in range(1 + (n + customer) % 3):
                description, unit_cents = CATALOG[(n + line * 2) % len(CATALOG)]
                conn.execute("""INSERT INTO invoice_lines (invoice_id, description, quantity, unit_cents)
                                VALUES (?, ?, ?, ?)""", (invoice_id, description, 1 + (n + line) % 8, unit_cents))
        totals = {row["invoice_id"]: row["amount_cents"] for row in conn.execute(
            "SELECT invoice_id, SUM(quantity * unit_cents) AS amount_cents FROM invoice_lines GROUP BY invoice_id")}
        for row in conn.execute("SELECT id, customer_id, due_on, status FROM invoices ORDER BY id").fetchall():
            n = row["id"] - 1
            if row["status"] == "void" or n % 7 == 3:                    # a void invoice, or one nobody has paid
                continue
            paid = datetime.strptime(row["due_on"], "%Y-%m-%d") + timedelta(days=(n % 11) - 4)
            if paid.strftime("%Y-%m-%d") > TODAY:
                continue
            part = totals[row["id"]] if n % 5 else totals[row["id"]] // 2       # every fifth invoice is half paid
            payment_id = conn.execute("""INSERT INTO payments (customer_id, received_at, amount_cents, method)
                                         VALUES (?, ?, ?, ?) RETURNING id""",
                                      (row["customer_id"], paid.strftime("%Y-%m-%d") + f"T{9 + n % 8:02d}:{n * 7 % 60:02d}",
                                       part, METHODS[n % 3])).fetchone()["id"]
            conn.execute("""INSERT INTO payment_applications (payment_id, invoice_id, amount_cents)
                            VALUES (?, ?, ?)""", (payment_id, row["id"], part))
    return conn


school = build_school(SCHOOL)
ledger = build_ledger(LEDGER)

ledger.execute("""
    CREATE VIEW invoice_balances AS
    SELECT invoices.id AS invoice_id, invoices.number, invoices.customer_id, invoices.issued_on,
           invoices.due_on, totals.amount_cents,
           totals.amount_cents - COALESCE(applied.paid_cents, 0) AS balance_cents
    FROM invoices
    JOIN (SELECT invoice_id, SUM(quantity * unit_cents) AS amount_cents
          FROM invoice_lines GROUP BY invoice_id) AS totals ON totals.invoice_id = invoices.id
    LEFT JOIN (SELECT invoice_id, SUM(amount_cents) AS paid_cents
               FROM payment_applications GROUP BY invoice_id) AS applied ON applied.invoice_id = invoices.id
    WHERE invoices.status = 'open'
""")

print("built", SCHOOL, "and", LEDGER, "| reports run as of", TODAY)


built scratch/school.db and scratch/ledger.db | reports run as of 2026-04-15


**1.** The grade distribution of 2025 Fall.


In [2]:
DISTRIBUTION = """
    SELECT enrollments.grade, grades.points, COUNT(*) AS students
    FROM enrollments
    JOIN sections ON sections.id = enrollments.section_id
    JOIN terms ON terms.id = sections.term_id
    LEFT JOIN grades ON grades.grade = enrollments.grade
    WHERE terms.name = ? AND enrollments.grade IS NOT NULL
    GROUP BY enrollments.grade
    ORDER BY grades.points DESC
"""

rows = school.execute(DISTRIBUTION, ("2025 Fall",)).fetchall()
most = max(row["students"] for row in rows)
for row in rows:
    print(f"{row['grade']:<3}{row['students']:>3}  {bar(row['students'], most, 24)}")


A    7  ###################
A-   6  ################
B+   8  #####################
B    9  ########################
B-   6  ################
C+   8  #####################
C    6  ################
C-   7  ###################
D    7  ###################
F    8  #####################
W    4  ###########


The grade is both what the query groups by and what it prints, and the points come from the `grades`
table only to order the bars from A downwards. `W` has no points, and `ORDER BY ... DESC` puts a
`NULL` last, which is where a withdrawal belongs. `bar` takes any pair of numbers, not only cents.


**2.** The instructors teaching the most students this term.


In [3]:
BUSIEST = """
    SELECT sections.instructor, COUNT(DISTINCT sections.id) AS sections,
           COUNT(enrollments.id) AS students
    FROM sections
    JOIN terms ON terms.id = sections.term_id
    LEFT JOIN enrollments ON enrollments.section_id = sections.id AND enrollments.status = 'enrolled'
    WHERE terms.name = ?
    GROUP BY sections.instructor
    ORDER BY students DESC, sections.instructor
"""

show(school.execute(BUSIEST, ("2026 Spring",)))


instructor    sections  students
------------  --------  --------
Dr. Ibarra    4         26      
Dr. Lindgren  4         21      
Dr. Hale      3         20      
Dr. Mensah    3         12      
(4 rows)


The join to `enrollments` multiplies a section's row by its students, so the sections have to be
counted with `COUNT(DISTINCT sections.id)` while the students are counted with a plain `COUNT`. The
`LEFT JOIN` keeps an instructor whose section nobody has taken, and `COUNT(enrollments.id)` scores
that section 0 rather than 1.


**3.** Rooms booked twice at the same time.


In [4]:
DOUBLE_BOOKED = """
    SELECT terms.name AS term, first.room, first.meets,
           first.starts_at || ' to ' || first.ends_at AS first_time,
           second.starts_at || ' to ' || second.ends_at AS second_time,
           first_course.code AS course, second_course.code AS clashes_with
    FROM sections AS first
    JOIN sections AS second ON second.id > first.id AND second.term_id = first.term_id
                           AND second.room = first.room AND second.meets = first.meets
                           AND first.starts_at < second.ends_at AND second.starts_at < first.ends_at
    JOIN courses AS first_course ON first_course.id = first.course_id
    JOIN courses AS second_course ON second_course.id = second.course_id
    JOIN terms ON terms.id = first.term_id
    ORDER BY terms.name, first.room, first.starts_at
"""

show(school.execute(DOUBLE_BOOKED))


term         room     meets  first_time      second_time     course   clashes_with
-----------  -------  -----  --------------  --------------  -------  ------------
2025 Fall    HUM 210  MWF    09:00 to 09:50  09:00 to 09:50  MAT-121  STA-200     
2025 Fall    HUM 302  MWF    11:00 to 11:50  11:00 to 11:50  CHE-110  HIS-110     
2025 Fall    LAB 101  MWF    08:00 to 08:50  08:00 to 08:50  MAT-120  PSY-101     
2025 Fall    LAB 101  MWF    14:00 to 14:50  14:00 to 14:50  MAT-120  PSY-101     
2025 Fall    MAT 004  MWF    09:00 to 09:50  09:00 to 09:50  BIO-101  ENG-105     
2025 Spring  HUM 302  MWF    08:00 to 08:50  08:00 to 08:50  MAT-121  STA-200     
2025 Spring  HUM 302  MWF    09:00 to 09:50  09:00 to 09:50  MAT-121  STA-200     
2025 Spring  MAT 004  TTh    13:00 to 14:15  13:00 to 14:15  MAT-120  PSY-101     
2025 Spring  SCI 118  TTh    09:30 to 10:45  09:30 to 10:45  CHE-110  HIS-110     
2025 Spring  SCI 210  MWF    08:00 to 08:50  08:00 to 08:50  BIO-101  ENG-105     
2025

The same self join as the timetable clash, with the room in place of the student: every pair of
sections in one term, kept when they share a room and a pattern of days and their times overlap.
`second.id > first.id` reports a pair once, and without it every clash would appear twice, once from
each side.


**4.** Invoiced per customer in the first quarter of 2026.


In [5]:
BY_CUSTOMER = """
    SELECT customers.name, COUNT(DISTINCT invoices.id) AS invoices,
           printf('%.2f', SUM(invoice_lines.quantity * invoice_lines.unit_cents) / 100.0) AS invoiced
    FROM invoices
    JOIN customers ON customers.id = invoices.customer_id
    JOIN invoice_lines ON invoice_lines.invoice_id = invoices.id
    WHERE invoices.status = 'open' AND invoices.issued_on >= :from AND invoices.issued_on < :to
    GROUP BY customers.id
    ORDER BY SUM(invoice_lines.quantity * invoice_lines.unit_cents) DESC
"""

show(ledger.execute(BY_CUSTOMER, {"from": "2026-01-01", "to": "2026-04-01"}))


name                invoices  invoiced
------------------  --------  --------
Aster Logistics     3         23175.00
Lantern Books       2         23045.00
Kestrel Media       3         12575.00
Northwind Foods     3         9605.00 
Pine Ridge Clinic   3         8300.00 
Granite Systems     2         7230.00 
Vela Robotics       3         7225.00 
Blue Harbor Design  3         3275.00 
(8 rows)


The quarter is a half-open range, so 31 March is in it and 1 April is not. The total has to be
ordered by the cents, not by the text `printf` made, since `'950.00'` sorts before `'9,500.00'` as
text. Void invoices are left out, as they are in every report that says what was invoiced.


**5.** Voiding an invoice, and what it does to the month.


In [6]:
MONTH_TOTAL = """
    SELECT printf('%.2f', SUM(invoice_lines.quantity * invoice_lines.unit_cents) / 100.0) AS invoiced
    FROM invoices JOIN invoice_lines ON invoice_lines.invoice_id = invoices.id
    WHERE invoices.status = 'open' AND strftime('%Y-%m', invoices.issued_on) = ?
"""

month = ledger.execute("SELECT strftime('%Y-%m', issued_on) AS month FROM invoices WHERE number = ?",
                       ("INV-1012",)).fetchone()["month"]
print(month, "invoiced before:", ledger.execute(MONTH_TOTAL, (month,)).fetchone()["invoiced"])
with transaction(ledger):
    changed = ledger.execute("UPDATE invoices SET status = 'void' WHERE number = ?", ("INV-1012",)).rowcount
print("rows changed:", changed)
print(month, "invoiced after: ", ledger.execute(MONTH_TOTAL, (month,)).fetchone()["invoiced"])


2025-11 invoiced before: 35615.00
rows changed: 1
2025-11 invoiced after:  23540.00


The invoice is marked, not deleted: its number and its lines stay, and every report that says
`status = 'open'` stops counting it. Deleting it would leave a hole in the numbering that nobody
could explain a year later, and would take its lines and any payment applied to it with it.


**6.** The customers who owe nothing.


In [7]:
PAID_UP = """
    SELECT customers.name, COUNT(payments.id) AS payments,
           printf('%.2f', COALESCE(SUM(payments.amount_cents), 0) / 100.0) AS paid_in_all
    FROM customers
    LEFT JOIN payments ON payments.customer_id = customers.id
    WHERE customers.id NOT IN (SELECT customer_id FROM invoice_balances WHERE balance_cents > 0)
    GROUP BY customers.id
    ORDER BY customers.name
"""

show(ledger.execute(PAID_UP))


name                  payments  paid_in_all
--------------------  --------  -----------
Harbor Lights Studio  0         0.00       
(1 rows)


The subquery lists the customers with something still owed, and `NOT IN` keeps the rest, including a
customer who has never been invoiced at all. The `LEFT JOIN` and `COALESCE` then report such a
customer as having paid nothing rather than leaving the column `NULL`. `NOT IN` is safe here because
`customer_id` in the view can never be `NULL`, and a `NULL` in that list would make `NOT IN` return
no rows at all.

Last, close both connections and remove the scratch folder:


In [8]:
school.close()
ledger.close()
shutil.rmtree("scratch")

print("scratch still there:", Path("scratch").exists())


scratch still there: False


---

&#8592; **Back to:** [Everyday Requests](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/sqlite3-deep-dive/20-everyday-requests.ipynb)  &nbsp;&middot;&nbsp;  [sqlite3, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlite3-deep-dive.html)
